In [1]:
import numpy as np
import pandas as pd
import argparse
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
import numpy as np
class Garnet:
    def __init__(self,nS=10,nA=5):
        self.nA = nA
        self.nS = nS
    def gen_probability(self):
        self.P = np.zeros((self.nA,self.nS,self.nS))
        for s in range(self.nS):
            for a in range(self.nA):
                mu,sigma = np.random.uniform(0,100),np.random.uniform(0,100)
                self.P[a,s,:] = np.random.normal(mu,sigma,self.nS)
                self.P[a,s] = np.exp(self.P[a,s])
                self.P[a,s] = self.P[a,s]/np.sum(self.P[a,s])
        return self.P
    def gen_reward(self):
        R = np.zeros((self.nA,self.nS,self.nS))
        self.exp_rew = self.gen_expected_reward()
        for a in range(self.nA):
            for s in range(self.nS):
                R[a,s,:] = self.exp_rew[s,a]
        return R
    def gen_cost(self):
        R = np.zeros((self.nA,self.nS,self.nS))
        self.exp_rew = self.gen_expected_constraint()
        for a in range(self.nA):
            for s in range(self.nS):
                R[a,s,:] = self.exp_rew[s,a]
        return R
    def gen_expected_reward(self):
        self.R = np.zeros((self.nS,self.nA))
        for s in range(self.nS):
            for a in range(self.nA):
                mu,sigma = np.random.uniform(0,10),np.random.uniform(0,10)
                self.R[s,a] = np.random.normal(mu,sigma)/10
        return self.R
    def gen_expected_constraint(self):
        self.R = np.zeros((self.nS,self.nA))
        for s in range(self.nS):
            for a in range(self.nA):
                mu,sigma = np.random.uniform(0,10),np.random.uniform(0,10)
                self.R[s,a] = np.random.normal(mu,sigma)/10
        return self.R

In [3]:
class Robust_pol_Kl_uncertainity:
    def __init__(self,nS,nA,cost_list,init_dist,alpha=1):
        self.nS = nS
        self.nA = nA
        self.cost_list = cost_list
        self.init_dist = init_dist
        self.gamma = 0.995
        self.alpha = alpha
    def set_cost(self,cost_list):
        self.cost_list = cost_list
    def calculate_infinite_Q(self,n,policy,P,s,a,C_KL):
        C = self.cost_list[n]
        Q = np.zeros((self.nS,self.nA))
        V = np.zeros(self.nS)
        tau = 100
        #print(self.init_dist)
        #s = np.random.choice(self.nS,p=self.init_dist)
        for t in range(tau):
            #print(policy[s])
            #a = np.random.choice(self.nA,p = policy[s])
            #print(P[a,s,:])
            next_state = np.random.choice(self.nS,p=P[a,s,:])
            #print("t=",self.t,a,s)
            P_star = np.array([P[a,s,i]*np.exp(self.alpha*V[i]/C_KL) for i in range(self.nS)])
            #print(P_star)
            #print("P_satr:",P_star)
            #print(P_star.shape)
            Q[s,a] = C[s,a] + self.gamma * np.dot(P_star,V)
            V = np.array([np.dot(policy[s,:],Q[s,:]) for s in range(self.nS)])
            #print("V=",V)
            #s = next_state
        #print(Q[s,a],V)
        return Q,V
    def evaluate_policy(self,policy,P,s,a,C_KL,n,t):
        self.t = t
        policy = np.array(policy)
        Q,V = self.calculate_infinite_Q(n, policy, P,s,a, C_KL)
        P_star = np.zeros((self.nS,self.nA,self.nS))
        #Pi_pi = torch.zeros((self.nS,self.nS,self.nA))
        Q_ = np.zeros((self.nS,self.nA))
        T = np.zeros((self.nS,self.nS))
        #for s in range(self.nS):
        #    for a in range(self.nA):
        P_star[s,a,:] = np.array([self.alpha*P_star[s,a,i]*np.exp(V[i]/C_KL) for i in range(self.nS)])
                #Q_[s,a] = self.cost_list[n][s,a] + self.gamma*torch.sum([P_star[s,a,s_next]*torch.sum([policy[s_next,a_next]*Q_[s_next,a_next] for a_next in range(self.nA)]) for s_next in range(self.nS)])#not correct
        for s in range(self.nS):
            for s_next in range(self.nS):
                T[s,s_next] = np.sum(np.array([policy[s,a]*P[a,s,s_next] for a in range(self.nA)]))
        I = np.eye(self.nS)
        Q_ = np.dot(np.linalg.inv(I-self.gamma*T),self.cost_list[n])
        d_pi = np.matmul(np.linalg.inv(I-self.gamma*T),self.init_dist)
        d_pi = d_pi/np.sum(d_pi)
        #print("d_pi:",d_pi)
        J = np.sum([self.init_dist[s]*np.sum([policy[s,a]*Q_[s,a] for a in range(self.nA)]) for s in range(self.nS)])
        #J_grad = np.array([d_pi[s]*np.array([Q_[s,a] for a in range(self.nA)]) for s in range(self.nS)])
        #print(J)
        return J

In [6]:
def discretize_budget(B,bins):
  return np.linspace(0,B,bins)
def one_hot(pol,nA):
  one_hot_pol = np.zeros((len(pol),nA))
  for i in range(len(pol)):
    one_hot_pol[i,int(pol[i])] = 1
  return one_hot_pol
class RVI:
  def __init__(self,env,args):
    self.C_d = discretize_budget(args.B,args.bins)
    self.eps = args.eps
    self.delta = args.delta
    self.beta = args.beta
    self.rho = args.rho
    self.eval = Robust_pol_Kl_uncertainity(args.nS,args.nA,args.cost_list,args.init_dist,args.alpha)
    self.pi = np.ones((args.nS,args.nA))*1/args.nA
    self.nS = args.nS
    self.nA = args.nA
    self.C_KL = 0.05
    self.V_g = np.ones((self.nS,args.bins))*(-args.B)
    self.V_r = np.zeros((self.nS,args.bins))
    self.Q_g = np.zeros((self.nS,args.bins,self.nA))
    self.Q_r = np.zeros((self.nS,args.bins,self.nA))
    self.gamma = args.gamma
    self.H = args.H
    self.N = args.N
    self.bins = args.bins
    self.init_state_dist = args.init_dist
    self.env  = env
    self.complete_init_dist = np.random.normal(0,1,self.nS*args.bins)
    self.complete_init_dist = np.exp(self.complete_init_dist)
    self.complete_init_dist = self.complete_init_dist/np.sum(self.complete_init_dist)
    self.save_dir = args.save_dir
  def collect_samples(self,s,a):
    samples=[];rewards=[];cost=[]
    P = self.env.gen_probability()
    R = self.env.gen_reward()
    C = self.env.gen_cost()
    #print("R=",R)
    #print("C=",C)
    for t in range(self.N):
      s_next = np.random.choice(self.nS,p=P[a,s,:])
      rewards.append(R[a,s,s_next])
      cost.append(C[a,s,s_next])
      samples.append(s_next)
    return samples,rewards,cost
  def cRVI(self):
    vf=[]
    cf=[]
    pi = np.ones((self.nS,self.bins,self.nA))*1/self.nA
    for h in tqdm(range(self.H-1,-1,-1)):
      P_0 = np.zeros((self.nA,self.nS,self.nS))
      for s in range(self.nS):
        for a in range(self.nA):
          samples,rewards,cost = self.collect_samples(s,a)
          for j in samples:
            P_0[a,s,j]+=1
          sum = np.sum(P_0[a,s,:])
          for s_next in range(self.nS):
            P_0[a,s,s_next] = P_0[a,s,s_next]/sum
          g_h = np.mean(cost)
          r_h = np.mean(rewards)
          for c in self.C_d:
            c_bin = int(np.digitize(c,self.C_d))-1
            #print(P_0[s,a,:])
            self.pi = pi[:,c_bin,:]
            self.pi = np.reshape(self.pi,(self.nS,self.nA))
            #print(self.pi 
            self.Q_g[s,c_bin,a] = self.eval.evaluate_policy(self.pi,P_0,s,a,self.C_KL,1,0)#need modification to robust value function
            self.Q_r[s,c_bin,a] = r_h+self.eval.evaluate_policy(self.pi,P_0,s,a,self.C_KL,0,0)#need modification to robust value function
            #self.V_g[s,c_bin] = np.max(self.Q_g[s,c,:])
            #self.V_r[s,c_bin] = np.max(self.Q_r[s,c,:])
            #print(self.Q_g[s,c_bin,a])
            #print(self.Q_r[s,c_bin,a])
      pi = np.zeros((self.nS,self.bins,self.nA))
      V_r = np.zeros((self.nS,self.bins))
      V_g = np.zeros((self.nS,self.bins))
      #print(V_r.shape)
      #print(V_g.shape)
      for s in range(self.nS):
        for c_bin in range(len(self.C_d)):
          c = np.digitize(self.C_d[c_bin],self.C_d)-1
          #a = np.argmax(self.Q_r[s,c,:])

          if(np.any(self.Q_g[s,c,:]>= -(self.H-h)*self.eps)):
            for a in range(nA):
              pi[s,c,a] = (self.H-h)*self.eps/self.Q_g[s,c,a]
          else:
            a = np.argmax(self.Q_r[s,c,:])
            pi[s,c,a] = 1.0
          pi[s,c,:] = np.exp(pi[s,c,:])
          sum = np.sum(pi[s,c,:])
          pi[s,c,:] = pi[s,c,:]/sum 
          '''if self.Q_g[s,c,a]>=-(self.H-h)*self.eps:
            self.V_g[s,c] = self.Q_g[s,c,a]
            self.V_r[s,c] = self.Q_r[s,c,a]
            self.pi[s] = a
          else:
            action_map={}
            flag = 0
            for a in range(self.nA):
              if self.Q_g[s,c,a]>=-(self.H-h)*self.eps:
                action_map[a] = self.Q_r[s,c,a]
                flag=1
            if flag==1:
              self.V_g[s,c] = self.Q_g[s,c,np.argmax(action_map)]
              self.V_r[s,c] = self.Q_r[s,c,np.argmax(action_map)]
              self.pi[s] = np.argmax(action_map)
            else:
              a = np.argmin(self.Q_g[s,c,:])
              self.V_g[s,c] = self.Q_g[s,c,a]
              self.V_r[s,c] = self.Q_r[s,c,a]
              self.pi[s] = a'''
          #print(self.Q_r[s,c,:])
          #print(pi[s,c,:])
          #print(V_r[s,c])
          #print(pi[s,c,:])
          #print(self.Q_r[s,c,:])
          V_r[s,c] = self.eval.evaluate_policy(pi[:,c,:],P_0,s,a,self.C_KL,1,0)
          V_g[s,c] = self.eval.evaluate_policy(pi[:,c,:],P_0,s,a,self.C_KL,1,0)
          #print(V_r[s,c])
          #print(V_g[s,c])
          #input()
      V_r_new = np.reshape(V_r,(self.nS*len(self.C_d),1))
      V_g_new = np.reshape(V_g,(self.nS*len(self.C_d),1))
      #print(V_r,V_g)
      J_r,J_c = np.dot(self.complete_init_dist,V_r_new),np.dot(self.complete_init_dist,V_g_new)
      #print(J_r,J_c)
      #input()
      vf.append(J_r)
      cf.append(J_c)
      #print(J_r,J_C
    df = {'vf':vf,'cf':cf}
    df = pd.DataFrame(df)
    df.to_excel(self.save_dir)
    return self.pi

In [7]:
if __name__=='__main__':
  env = Garnet()
  nS,nA = env.nS,env.nA
  cost_list = [env.gen_expected_reward(),env.gen_expected_constraint()]
  init_dist = np.random.normal(0,1,nS)
  init_dist = np.exp(init_dist)
  init_dist = init_dist/np.sum(init_dist)
  #print(init_dist)
  #print(np.sum(init_dist))
  parser = argparse.ArgumentParser("The required hyperparameters for cRCMDP")
  parser.add_argument('--B',type=int,default=15,help='budget')
  parser.add_argument('--bins',type=int,default=20,help='number of bins')
  parser.add_argument('--eps',type=float,default=0.01,help='epsilon')
  parser.add_argument('--delta',type=float,default=0.05,help='delta')
  parser.add_argument('--beta',type=float,default=0.01,help='beta')
  parser.add_argument('--rho',type=float,default=0.01,help='rho')
  parser.add_argument('--gamma',type=float,default=0.995,help='discount factor')
  parser.add_argument('--H',type=int,default=1000,help='horizon')
  parser.add_argument('--N',type=int,default=1000,help='number of samples')
  parser.add_argument('--nS',type=int,default=nS,help='number of states')
  parser.add_argument('--nA',type=int,default=nA,help='number of actions')
  parser.add_argument('--cost_list',type=list,default=cost_list,help='cost list')
  parser.add_argument('--init_dist',type=list,default=init_dist,help='initial distribution')
  parser.add_argument('--alpha',type=float,default=0.00001,help='alpha')
  parser.add_argument('--save_dir',type=str,default='./aistats_VI_cRCMDP_Garnet.xlsx',help='save directory')
  args, unknown = parser.parse_known_args()
  rvi = RVI(env,args)
  pi = rvi.cRVI()
  print(pi)

100%|██████████| 1000/1000 [3:25:53<00:00, 12.35s/it] 


[[0.22524103 0.21359155 0.19947383 0.18648733 0.17520626]
 [0.39545654 0.24756103 0.1473147  0.11599316 0.09367457]
 [0.32983752 0.23475038 0.16032907 0.14557242 0.12951062]
 [0.42539833 0.20419672 0.16783115 0.12386491 0.0787089 ]
 [0.29747935 0.26672652 0.17086281 0.14772197 0.11720935]
 [0.2740885  0.21152148 0.19350215 0.17822297 0.1426649 ]
 [0.30687074 0.21603658 0.17187806 0.16128515 0.14392947]
 [0.41040026 0.27541794 0.19130201 0.08990659 0.0329732 ]
 [0.25091523 0.23679129 0.20233341 0.17390599 0.13605409]
 [0.59793303 0.22189294 0.0977508  0.05138115 0.03104208]]
